In [0]:
catalog = "logistics"
gold_schema = f"{catalog}.gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

In [0]:
# Read gold star schema: dimensions and facts
dim_carrier_df = spark.table(f"{gold_schema}.dim_carrier")
dim_warehouse_df = spark.table(f"{gold_schema}.dim_warehouse")
fact_route_df = spark.table(f"{gold_schema}.fact_route")
fact_shipment_df = spark.table(f"{gold_schema}.fact_shipment")
fact_delivery_attempt_df = spark.table(f"{gold_schema}.fact_delivery_attempt")

# Status history has no gold fact equivalent, so it is still read from silver
silver_shipment_status_history_df = spark.table(f"{catalog}.silver.silver_shipment_status_history")


In [0]:
print(f"dim_carrier: {dim_carrier_df.count()} rows")
print(f"dim_warehouse: {dim_warehouse_df.count()} rows")
print(f"fact_route: {fact_route_df.count()} rows")
print(f"fact_shipment: {fact_shipment_df.count()} rows")
print(f"fact_delivery_attempt: {fact_delivery_attempt_df.count()} rows")
print(f"silver_shipment_status_history: {silver_shipment_status_history_df.count()} rows")

In [0]:
from pyspark.sql import functions as F

# Referential integrity: every fact foreign key must resolve to its dimension
fk_checks = [
    ("fact_shipment.carrier_sk", fact_shipment_df, "carrier_sk", dim_carrier_df, "carrier_sk"),
    ("fact_shipment.origin_warehouse_sk", fact_shipment_df, "origin_warehouse_sk", dim_warehouse_df, "warehouse_sk"),
    ("fact_route.origin_warehouse_sk", fact_route_df, "origin_warehouse_sk", dim_warehouse_df, "warehouse_sk"),
    ("fact_delivery_attempt.carrier_sk", fact_delivery_attempt_df, "carrier_sk", dim_carrier_df, "carrier_sk"),
]
for name, fact_df, fk, dim_df, pk in fk_checks:
    orphans = fact_df.join(dim_df.select(F.col(pk).alias(fk)), fk, "left_anti").count()
    assert orphans == 0, f"{name}: {orphans} orphan keys"
    print(f"{name}: OK")

In [0]:
from pyspark.sql import functions as F

def write_gold(df, table_name):
    """Run a completeness check, then overwrite as a Delta table in the gold schema."""
    null_counts = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).first().asDict()
    bad = {c: n for c, n in null_counts.items() if n}
    if bad:
        raise ValueError(f"{table_name} failed completeness check: {bad}")
    full_name = f"{gold_schema}.{table_name}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(full_name))
    print(f"{full_name}: {spark.table(full_name).count()} rows")

In [0]:
# 1. Shipments by status
gold_shipments_by_status_df = (fact_shipment_df.groupBy("shipment_status").agg(
    F.count("*").alias("shipment_count"),
    F.round(F.sum("shipping_cost"), 2).alias("total_shipping_cost"),
    F.round(F.avg("shipping_cost"), 2).alias("avg_shipping_cost"),
    F.round(F.avg("package_weight_kg"), 2).alias("avg_weight_kg")))
write_gold(gold_shipments_by_status_df, "gold_shipments_by_status")

# 2. Carrier performance
gold_carrier_performance_df = (fact_shipment_df.join(dim_carrier_df.select("carrier_sk", "carrier_id", "carrier_name", "carrier_tier", "service_region"), "carrier_sk", "inner")
    .groupBy("carrier_id", "carrier_name", "carrier_tier", "service_region").agg(
        F.count("*").alias("shipment_count"),
        F.round(F.sum("shipping_cost"), 2).alias("total_revenue"),
        F.round(F.sum("package_weight_kg"), 2).alias("total_weight_kg"),
        F.sum(F.when(F.col("shipment_status") == "DELIVERED", 1).otherwise(0)).alias("delivered_count"))
    .withColumn("avg_cost_per_kg", F.coalesce(F.round(F.col("total_revenue") / F.col("total_weight_kg"), 4), F.lit(0.0))))
write_gold(gold_carrier_performance_df, "gold_carrier_performance")

# 3. Daily shipments
gold_daily_shipments_df = (fact_shipment_df.withColumn("shipment_date", F.to_date("created_ts"))
    .groupBy("shipment_date").agg(
        F.count("*").alias("shipment_count"),
        F.round(F.sum("shipping_cost"), 2).alias("total_revenue"),
        F.round(F.sum("package_weight_kg"), 2).alias("total_weight_kg")))
write_gold(gold_daily_shipments_df, "gold_daily_shipments")

# 4. Destination city summary
gold_destination_city_summary_df = (fact_shipment_df.groupBy("destination_city").agg(
    F.count("*").alias("shipment_count"),
    F.round(F.avg("shipping_cost"), 2).alias("avg_shipping_cost"),
    F.round(F.avg("package_weight_kg"), 2).alias("avg_weight_kg")))
write_gold(gold_destination_city_summary_df, "gold_destination_city_summary")

# 5. Service level summary
gold_service_level_summary_df = (fact_shipment_df.groupBy("service_level").agg(
    F.count("*").alias("shipment_count"),
    F.round(F.avg("shipping_cost"), 2).alias("avg_shipping_cost"),
    F.round(F.avg("package_weight_kg"), 2).alias("avg_weight_kg")))
write_gold(gold_service_level_summary_df, "gold_service_level_summary")

In [0]:
is_success = F.when(F.col("attempt_result") == "DELIVERED", 1).otherwise(0)

# 6. Attempt result / failure reason summary
gold_delivery_attempt_summary_df = (fact_delivery_attempt_df.groupBy("attempt_result", "failure_reason").agg(
    F.count("*").alias("attempt_count"),
    F.countDistinct("shipment_id").alias("shipment_count")))
write_gold(gold_delivery_attempt_summary_df, "gold_delivery_attempt_summary")

# 7. City delivery success
gold_city_delivery_success_df = (fact_delivery_attempt_df.groupBy("delivery_city").agg(
        F.count("*").alias("attempt_count"),
        F.sum(is_success).alias("successful_attempts"))
    .withColumn("success_rate_pct", F.coalesce(F.round(F.col("successful_attempts") * 100.0 / F.col("attempt_count"), 2), F.lit(0.0))))
write_gold(gold_city_delivery_success_df, "gold_city_delivery_success")

In [0]:
# 8. Status transitions (only shipments that have a tracking event)
gold_status_transitions_df = (silver_shipment_status_history_df
    .filter(F.col("cdc_operation") != "NO_EVENT")
    .groupBy(F.col("prev_event_type").alias("from_status"), F.col("event_type").alias("to_status"))
    .agg(F.count("*").alias("transition_count")))
write_gold(gold_status_transitions_df, "gold_status_transitions")

# 9. CDC operation summary
total_shipments = silver_shipment_status_history_df.count()
gold_cdc_operation_summary_df = (silver_shipment_status_history_df.groupBy("cdc_operation")
    .agg(F.count("*").alias("shipment_count"))
    .withColumn("pct_of_total", F.round(F.col("shipment_count") * 100.0 / F.lit(total_shipments), 2)))
write_gold(gold_cdc_operation_summary_df, "gold_cdc_operation_summary")

In [0]:
# Summary of gold tables
gold_tables = ["gold_shipments_by_status", "gold_carrier_performance", "gold_daily_shipments",
    "gold_destination_city_summary", "gold_service_level_summary", "gold_delivery_attempt_summary",
    "gold_city_delivery_success", "gold_status_transitions", "gold_cdc_operation_summary"]

rows = []
for t in gold_tables:
    df = spark.table(f"{gold_schema}.{t}")
    total_nulls = df.select(sum(F.sum(F.col(c).isNull().cast("int")) for c in df.columns).alias("n")).first()["n"]
    fmt = spark.sql(f"DESCRIBE DETAIL {gold_schema}.{t}").first()["format"]
    assert total_nulls == 0, f"{t} failed completeness check"
    rows.append((t, fmt, df.count()))

display(spark.createDataFrame(rows, "table_name string, format string, row_count long"))